In [ ]:
folder = '23Jul06-1101'

In [ ]:
# load
from collections import defaultdict
from tqdm import tqdm
import cloudpickle
import pathlib

folder = pathlib.Path(folder)
assert folder.exists()

def tree():
    return defaultdict(tree)


pval_seed_method_ana_effect = tree()

for file in tqdm(folder.glob('analysis*.p'), desc='load per experiment'):
    with open(file, 'rb') as f:
        pval, seed, method, ana, effect = cloudpickle.load(file=f)
        pval_seed_method_ana_effect[pval][seed][method] = ana, effect

In [ ]:
import hrba

def get_max_f1(ana, effect):
    # gets max f1 score from all regions in first epoch
    f1 = hrba.graph.get_f1(mask=effect.mask, 
                           mask_idx=ana.exp.mask_idx, 
                           children= ana.epoch_list[0].child_dict[0])
    return max(f1)

In [ ]:
import numpy as np
from sklearn.metrics import f1_score, recall_score, confusion_matrix


# extract
pval_list = sorted(pval_seed_method_ana_effect.keys())
max_seed = max(max(pval_seed_method_ana_effect[p].keys()) for p in pval_list)

shape = max_seed + 1, len(pval_list)
score_dict = defaultdict(lambda: np.full(shape=shape, fill_value=np.nan))

for pval, t0 in pval_seed_method_ana_effect.items():
    for seed, t1 in t0.items():
        for method, (ana, effect) in t1.items():
            # build mask of predicted area (union of all effect masks)
            mask_pred = np.zeros(ana.exp.mask_idx.shape, dtype=bool)
            for _effect in ana.effect_tup:
                mask_pred |= _effect.mask

            # build y_true / y_pred in sklearn format
            mask_active = ana.exp.mask_idx > -1
            y_true = effect.mask[mask_active]
            y_pred = mask_pred[mask_active]
            
            # compute scores
            _f1 = f1_score(y_true=y_true, y_pred=y_pred, zero_division=0)
            _sens = recall_score(y_true=y_true, y_pred=y_pred, zero_division=0)
            conf_mat = confusion_matrix(y_true=y_true, y_pred=y_pred)

            # store
            pval_idx = pval_list.index(pval)
            score_dict[method, 'f1'][seed, pval_idx] = _f1
            score_dict[method, 'sens'][seed, pval_idx] = _sens
            score_dict[method, 'spec'][seed, pval_idx] = \
                conf_mat[0, 0] / (conf_mat[0, 0] + conf_mat[1, 0])

In [ ]:
# plot
import seaborn as sns
import matplotlib.pyplot as plt

sns.set()

fig, ax = plt.subplots(2, 3)
style_dict = {'AnalysisTFCE': '--', 'AnalysisHRBA': '-'}
for _ax, feat in zip(ax[0, :], ('f1', 'sens', 'spec')):
    plt.sca(_ax)
    for method, style in style_dict.items():
        line = plt.plot(pval_list, score_dict[method, feat].T, linestyle=style, color='k')

    plt.xlabel('pval')
    plt.ylabel(feat)
    plt.xscale('log')
    if feat == 'f1':
        # plot max f1 score
        plt.plot(pval_list, score_dict['HRBA_max', feat].T, linestyle=style, color='r')

for _ax, feat in zip(ax[1, :], ('f1', 'sens', 'spec')):
    plt.sca(_ax)
    x = score_dict['AnalysisHRBA', feat] - score_dict['AnalysisTFCE', feat]
    plt.plot(pval_list, x.T, linestyle=style)

    plt.xlabel('pval')
    plt.ylabel(f'{feat}: HRBA - TFCE')
    plt.xscale('log')
    
fig.set_size_inches(15, 5)
fig.tight_layout()

# Examine Worst Cases
HRBA does poorest as compared to TFCE

In [ ]:
import hrba.plot
diff = score_dict['AnalysisHRBA', 'f1'] - score_dict['AnalysisTFCE', 'f1']

for idx in np.argsort(diff.flatten())[:1]:
    # lookup analysis & effect
    seed, pval_idx = np.unravel_index(np.nanargmin(diff), diff.shape)
    pval = pval_list[pval_idx]
    ana, effect = pval_seed_method_ana_effect[pval][seed]['AnalysisHRBA']

    # lookup / print f1 scores
    f1_hrba = score_dict['AnalysisHRBA', 'f1'][seed, pval_idx]
    f1_tfce = score_dict['AnalysisTFCE', 'f1'][seed, pval_idx]
    print(f'\n\nseed {seed} pval: {pval:.2E} HRBA f1: {f1_hrba:.3f} TFCE f1: {f1_tfce:.3f}')
    
    # plot
    plt.figure()
    epoch = ana.epoch_list[0]
    fig = hrba.plot.scatter_size_vs_two(epoch=epoch, mask=effect.mask)
    fig.set_size_inches(15, 5)
    plt.show()

# Modelling LLR as a function of size under $H_0$

In [ ]:
# strip to h0 (no permute data may not belong to h0)
llr = epoch.llr[1:, :]
size = epoch.size[1:, :]

for _size in (1, 2, 4, 8, 16, 32, size.max()):
    _llr = llr[size == _size]
    fig = plt.figure()
    plt.hist(_llr, bins=40, density=True)
    # plt.axvline(lin_reg.predict(np.array(_size).reshape(-1, 1))[0], color='r')
    fig.set_size_inches(10, 2)
    plt.suptitle(f'{_size} voxel regions')
plt.xlabel('LLR');

In [ ]:
plt.hist(np.log10(size).flatten(), bins=20);
plt.yscale('log')
plt.xlabel('size (voxels)')
plt.ylabel('count')
plt.suptitle('region size');

In [ ]:
from sklearn.linear_model import LinearRegression

lin_reg = LinearRegression(fit_intercept=True)
lin_reg.fit(X=size.reshape(-1, 1), y=llr.flatten(), sample_weight=size.flatten());
lin_reg.coef_, lin_reg.intercept_

plt.scatter(size.flatten(), llr.flatten(), alpha=.1, label='observed')
x = np.geomspace(size.min(), size.max(), 101)
llr_predict = lin_reg.predict(x.reshape(-1, 1))
plt.plot(x, llr_predict, linewidth=2, color='r', label='predicted')
plt.legend()
plt.xlabel('size')
plt.ylabel('llr')

plt.figure()
error = llr - lin_reg.predict(size.reshape(-1, 1)).reshape(size.shape)
plt.scatter(size.flatten(), error.flatten(), alpha=.1, label='error')
plt.xlabel('size (voxels)')
plt.ylabel('error')
plt.savefig('fernando_error.png')

In [ ]:
from scipy.optimize import minimize


_size = epoch.size[1:, :]
x = np.stack([np.ones(_size.size), _size.flatten()]).T
y = epoch.llr[1:, :].flatten()

def obj(theta):
    """ negative log likelihood of model """
    theta = theta.reshape((2, 2))
    error = y - x @ theta[0, :]
    sigma = x @ theta[1, :]
    
    return 1/2 * ((error / sigma ** .5) ** 2).sum() + np.log(sigma ** .5).sum()

res = minimize(obj, x0=np.ones(4), method='Nelder-Mead')
res

In [ ]:
def llr_to_z(theta, llr, size):
    theta = theta.reshape((2, 2))
    x = np.stack([np.ones(size.size), size.flatten()]).T
    error = llr.flatten() - x @ theta[0, :]
    sigma = x @ theta[1, :]
    return (error / sigma ** .5).reshape(llr.shape)

In [ ]:
z = llr_to_z(theta=res.x, llr=epoch.llr, size=epoch.size)

In [ ]:
hrba.plot.scatter_size_vs_stat(epoch=epoch, mask=effect.mask, y_feat=z)

thresh = np.percentile(z.max(axis=1), [95])
plt.axhline(thresh, color='r', linestyle='--')